In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
from datetime import datetime
import pickle

In [3]:
df = pd.read_csv("processed_data.csv")

## **Feature Engineering**

### Drop unnecessary columns

In [5]:
df = df.drop(columns=['track_id', 'track_name', 'artists', 'album_name'])

In [6]:
df['duration_min'] = df['duration_ms'] / 60000

### Creating Mood Categories from Audio Features

This section creates a new feature called `mood_category` based on the combination of energy and valence scores. 

According to Spotify audio feature definitions:
- Energy reflects the intensity and activity level of a track.
- Valence reflects the musical positivity or emotional tone.

By combining these two variables, songs can be categorized into different emotional moods to support listener behavior analysis and practical interpretation.

In [7]:
def mood_category(row):
    if row['energy'] > 0.6 and row['valence'] > 0.6:
        return 'Happy/Energetic'
    
    elif row['energy'] < 0.4 and row['valence'] < 0.4:
        return 'Sad/Chill'
    
    elif row['energy'] > 0.6 and row['valence'] < 0.4:
        return 'Aggressive'
    
    else:
        return 'Relaxing'

df['mood_category'] = df.apply(mood_category, axis=1)



### Scale loudness

In [8]:
scaler = StandardScaler()

df['loudness_scaled'] = scaler.fit_transform(df[['loudness']])

### Categorize tempo

In [9]:
### categorize tempo into bins: slow, medium, fast

df['tempo_category'] = pd.cut(df['tempo'], bins=[-1, 90, 140, df['tempo'].max()], labels=['slow', 'medium', 'fast'])

### Encode categorical features

In [10]:
df['explicit'] = df['explicit'].astype(int) 

In [11]:
# convert to string for encoding
df['tempo_category'] = df['tempo_category'].astype('str')

# one-hot encoding for 'tempo_category'
df = pd.get_dummies(df, columns=['tempo_category'], drop_first=True)

### Drop columns that has been transformed into other columns

In [12]:
df = df.drop(columns=['duration_ms', 'danceability', 'loudness', 'tempo','track_genre'])

In [13]:
df.head()

,popularity,explicit,energy,key,mode,speechiness,acousticness,instrumentalness,liveness,valence,time_signature,duration_min,mood_category,loudness_scaled,tempo_category_medium,tempo_category_slow
0,73,0,0.4610,1,0,0.1430,0.0322,0.000001,0.3580,0.715,4,3.844433,Relaxing,0.298800,False,True
1,55,0,0.1660,1,1,0.0763,0.9240,0.000006,0.1010,0.267,4,2.493500,Sad/Chill,-1.794228,False,True
2,57,0,0.3590,0,1,0.0557,0.2100,0.000000,0.1170,0.120,4,3.513767,Sad/Chill,-0.297440,False,True
3,71,0,0.0596,0,1,0.0363,0.9050,0.000071,0.1320,0.143,3,3.365550,Sad/Chill,-2.049645,False,False
4,82,0,0.4430,2,1,0.0526,0.4690,0.000000,0.0829,0.167,4,3.314217,Relaxing,-0.286864,True,False


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 113549 entries, 0 to 113548
Data columns (total 16 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   popularity             113549 non-null  int64  
 1   explicit               113549 non-null  int64  
 2   energy                 113549 non-null  float64
 3   key                    113549 non-null  int64  
 4   mode                   113549 non-null  int64  
 5   speechiness            113549 non-null  float64
 6   acousticness           113549 non-null  float64
 7   instrumentalness       113549 non-null  float64
 8   liveness               113549 non-null  float64
 9   valence                113549 non-null  float64
 10  time_signature         113549 non-null  int64  
 11  duration_min           113549 non-null  float64
 12  mood_category          113549 non-null  str    
 13  loudness_scaled        113549 non-null  float64
 14  tempo_category_medium  113549 non-null  bool   

In [16]:
df.to_csv('cleaned_data.csv', index=False)